In [2]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import zipfile

In [3]:
transform = transforms.ToTensor()

with zipfile.ZipFile("C:\\Users\\Marien\\Downloads\\archive.zip", "r") as zip_ref:
    zip_ref.extractall("cats")

dataset = torchvision.datasets.ImageFolder(root="cats", transform=transform)

data_loader = DataLoader(dataset, batch_size=64, shuffle=True)

batch_size = 64

In [ ]:
class Resnet(nn.Module):
    def __init__(self, n_in, n_out, n_group):
        super().__init__()
        self.n_in = n_in
        self.n_out = n_out
        self.n_group = n_group

        self.Resnet = nn.Sequential(
            nn.GroupNorm(self.n_group, self.n_in),
            nn.SiLU(),
            nn.Conv2d(self.n_in, self.n_out, 3, 1, 1) #H * W -> H * W
            )

        self.skip = nn.Conv2d(self.n_in, self.n_out, 1, 1, 0)

    def forward(self, x):
        return self.Resnet(x) + self.skip(x)
    
class Downsample(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.n_in = n_in
        self.n_out = n_out

        self.downsample = nn.Conv2d(self.n_in, self.n_out, 3, 2, 1) # H*W -> H/2 * W/2 

    def forward(self, x):
        return self.downsample(x)

class Upsample(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.n_in = n_in
        self.n_out = n_out

        self.upsample = nn.ConvTranspose2d(self.n_in, self.n_out, 4, 2, 1) # H*W -> H*2 * W*2 

    def forward(self, x):
        return self.upsample(x)

class TimeEmbedding(nn.Module):
    def __init__(self, d ,channels):
        super().__init__()
        self.d = d
        self.mlp = nn.Sequential(
            nn.Linear(2 * d, d),
            nn.SiLU(),
            nn.Linear(d, channels)
        )

    def embedding(self, t):
        puls = [2 * torch.pi / i for i in range(1, self.d + 1)]
        embed = []
        for pul in puls:
            embed.append(torch.sin(pul * t))
            embed.append(torch.cos(pul * t))
        embed = torch.stack(embed) #convertit la liste en un tenseur
        return embed

    def forward(self, t):
        embed = self.embedding(t)
        embed = self.mlp(embed)
        return embed

In [ ]:
class Diffusion_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding64 = TimeEmbedding(64, 64)
        self.embedding128 = nn.Linear(64, 128)
        self.embedding256 = nn.Linear(64, 256)
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1)
        self.resnet1 = Resnet(64, 64, 8)
        self.downsample1 = Downsample(64, 128)
        self.resnet2 = Resnet(128, 128, 8)
        self.downsample2 = Downsample(128, 256)
        self.resnet3 = Resnet(256, 256, 8)
        self.upsample1 = Upsample(256, 128)
        self.resnet4 = Resnet(256, 128, 8)
        self.upsample2 = Upsample(128, 64) 
        self.resnet5 = Resnet(128, 64, 8)
        self.convfinal = nn.Conv2d(64, 3, 3, 1, 1)

    def forward(self, xt, t):
        t_embed = self.embedding64(t) #embedding de t, tensor de dimension 64
        t_embed = t_embed.unsqueeze(-1).unsqueeze(-1) #tensor de dimension 64*1*1

        xt_t = self.conv1(xt) # H*W -> H*W
        xt1 = self.resnet1(xt_t) + t_embed # H*W -> H*W, 8: nombre de groupes pour GroupNorm, (B_size, 64, 64, 64)
        xt2 = self.downsample1(xt1) # H*W -> H/2 * W/2 (B_size, 128, 32, 32)
        t_embed1 = self.embedding128(t_embed.squeeze(-1).squeeze(-1)) #embedding de t, tensor de dimension 128
        t_embed1 = t_embed1.unsqueeze(-1).unsqueeze(-1) #tensor de dimension 128*1*1
        xt3 = self.resnet2(xt2) + t_embed1 # H/2 * W/2 -> H/2 * W/2 (B_size, 128, 32, 32)
        xt4 = self.downsample2(xt3) # H/2 * W/2 -> H/4 * W/4 (B_size, 256, 16, 16)
        t_embed2 = self.embedding256(t_embed.squeeze(-1).squeeze(-1)) #embedding de t, tensor de dimension 256
        t_embed2 = t_embed2.unsqueeze(-1).unsqueeze(-1) #tensor de dimension 256*1*1
        xt5 = self.resnet3(xt4) + t_embed2 #Bottleneck, H/4 * W/4 -> H/4 * W/4 (B_size, 256, 16, 16)
        xt6 = self.upsample1(xt5) # H/4 * W/4 -> H/2 * W/2 (B_size, 128, 32, 32)
        concat = torch.cat([xt6, xt3], dim=1) # H/2 * W/2 -> H/2 * W/2 (B_size, 256, 32, 32)
        t_embed3 = self.embedding128(t_embed.squeeze(-1).squeeze(-1)) #embedding de t, tensor de dimension 128
        t_embed3 = t_embed3.unsqueeze(-1).unsqueeze(-1) #tensor de dimension 128*1*1
        xt7 = self.resnet4(concat) + t_embed3 # H/2 * W/2 -> H/2 * W/2 (B_size, 128, 32, 32)
        xt8 = self.upsample2(xt7) # H/2 * W/2 -> H * W
        concat2 = torch.cat([xt8, xt1], dim=1) # H * W -> H * W (B_size, 128, 64, 64)
        t_embed4 = self.embedding64(t_embed.squeeze(-1).squeeze(-1)) #embedding de t, tensor de dimension 64
        t_embed4 = t_embed4.unsqueeze(-1).unsqueeze(-1) #tensor de dimension 64*1*1
        xt9 = self.resnet5(concat2) + t_embed4 # H * W -> H * W
        xt10 = self.convfinal(xt9) # H * W -> H * W
        return xt10

In [5]:
T = 1000

def beta_schedule(timesteps):
    beta_start = 0.0001
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, timesteps)

beta = beta_schedule(T)
alpha = 1 - beta
alpha_hat = torch.cumprod(alpha, dim=0)

In [ ]:
model = Diffusion_model()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.MSELoss()

In [ ]:
for epoch in range(10):
    for i, (images, _) in enumerate(data_loader):
        x0 = images
        optimizer.zero_grad()
        index = torch.randint(0, T, (batch_size,))
        alpha_tensor = alpha_hat[index] #tenseur des alpha_hat_t de taille batch_size
        noise = torch.randn_like(x0) #tenseur de bruit de taille batch_size
        xt = torch.sqrt(alpha_tensor) * x0 + torch.sqrt(1 - alpha_tensor) * noise #tenseur de taille batch_size
        predicted_noise = model(xt, index) #tenseur de taille batch_size
        Loss = criterion(predicted_noise, noise)
        Loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch + 1}/10], Loss: {Loss.item():.4f}")
    


